In this notebook, I will interpret cross validation and bayesian optimization results for a variety of combinations of models and response variables

In [4]:
import pickle
import numpy as np
import pandas as pd
from pyhere import here

In [ ]:
rmse_value = 0
imputer_chosen = ""
params = ""
scaler = ""
example_results_dict_visualization = {
    'CVscore': np.float(rmse_value),
    'best_params': {
        'impute': imputer_chosen,
        'params(condensed)': params
        'scale': scaler
    },
    'logCVscore': np.float(rmse_value),
    'logbest_params': {
        'impute': imputer_chosen,
        'params(condensed)': params
        'scale': scaler
    }
}

with open(here("pipeline/results", "rf_LRATIO_results.pkl"), 'rb') as file:
    rf_lrat_results = pickle.load(file)
print(rf_lrat_results)

{'CVscore': np.float64(-77.59716309629432), 'best_params': OrderedDict({'impute': KNNImputer(), 'model__bootstrap': False, 'model__max_depth': 9, 'model__max_features': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 3, 'model__n_estimators': 887, 'scale': 'passthrough'}), 'logCVscore': np.float64(-0.027990054341194476), 'logbest_params': OrderedDict({'impute': KNNImputer(), 'model__bootstrap': False, 'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 790, 'scale': 'passthrough'})}


In [6]:
print(list(rf_lrat_results['best_params'])[1:-1])

['model__bootstrap', 'model__max_depth', 'model__max_features', 'model__min_samples_leaf', 'model__min_samples_split', 'model__n_estimators']


Due to the setup of the results dictionary from the pickle file, in theory I can extract everything I need using dictionary methods like indexing and stuff, loop through all of them with a for loop and put those in a pandas dataframe of the results. Then, I can just sort by response variable, display the cv error and fit the best params.

The hard part will be getting the original pipeline to put the params to, but that should work with some f-string replacement working backwards.

In [ ]:
results_list = []

for response in ["LRATIO", "LM", "L_BOL", "MASS", "DIAM", "SURF_DENS", "TEMP", "T_BOL"]:
    for space in ["xgblogratio_space.pkl", "rf_space.pkl", "rflogratio_space.pkl", "catboost_space.pkl", "xgboost_space.pkl", "tree_space.pkl", "catlogratio_space.pkl"]:
        with open(here("pipeline/results", f"{response}_{space}.pkl"), 'wb') as file:
            results = pickle.load(file)
            space_name = space.split("_")[0]

            param_values = []
            for param in list(results['best_params'])[1:-1]:
                param_values.append(results['best_params'][param])

            log_param_values = []
            for param in list(results['log_best_params'])[1:-1]:
                log_param_values.append(results['log_best_params'][param])

            row1 = {
                'log': 0,
                'var': response,
                'space': space_name,
                'CV_RMSE': results['CVscore'],
                'impute_strat': results['best_params']['impute'],
                'scale_strat': results['best_params']['scale'],
                'param_names': list(results['best_params'])[1:-1], 
                'param_values': param_values
            }
            row2 = {
                'log' = 1,
                'var': response,
                'space': space_name,
                'CV_RMSE': results['logCVscore'],
                'impute_strat': results['logbest_params']['impute'],
                'scale_strat': results['logbest_params']['scale'],
                'param_names': list(results['logbest_params'])[1:-1],
                'param_values': log_param_values
            }

            results_list.append(row1)
            results_list.append(row2)

results_df = pd.DataFrame(dict_list)


Once i get my results back, I can plug them into this to make them a list. then, I can easily find the best model for each response variable and trace that back to the pipeline used to fit it and fit that on the whole training set if I want to, to get final values for metrics/such.